# 🐧 Linux Systems, Shell Scripting & Secured Shell (SSH) Masterclass
### 🎓 Hands-On Interactive Laboratory Guide | Industry Training Module
**Institution:** GLA University, Department of Computer Engineering & Applications  
**Module:** High Performance Computing & Cloud Infrastructure Foundation  
**Level:** Basic to Medium (Applied Foundations & Automation)  

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shivam990q/GLA-HPC-Masterclass/blob/main/COLAB_NOTEBOOKS/Day_01_Linux_Basics_Shell_Scripting_and_SSH_Colab.ipynb)

---

### 🎯 Session Learning Objectives
By completing this interactive laboratory, students and practitioners will master:
1. **Linux Filesystem Architecture:** Navigating directories, understanding standard hierarchies (`/home`, `/etc`, `/tmp`, `/scratch`), and mastering file permissions (`chmod`, `chown`).
2. **Core Terminal Commands & Text Manipulation:** Using pipelines (`|`), redirects (`>`, `>>`), and stream filters (`grep`, `find`, `sort`, `uniq`, `wc`, `awk`).
3. **Environment Setup & Configuration:** Managing shell variables, export scopes, `$PATH` resolution, and `.bashrc` startup configuration.
4. **Shell Scripting for Automation (Basic to Medium):** Writing scripts with shebang lines, arguments (`$1`, `$2`), conditionals (`if-elif-else`), loops (`for`, `while`), functions, and defensive scripting (`set -euo pipefail`).
5. **Secured Shell (SSH) & Cluster Connectivity:** Generating Ed25519 cryptographic keypairs, managing `authorized_keys`, configuring `~/.ssh/config` host aliases, and performing secure data transfers (`scp`, `rsync`).

> **💡 Colab Execution Note:** Each code block uses the `%%bash` cell magic. Click the **Play button ▶** on the left of each cell to execute the commands directly in your Linux cloud environment!

---
## 📁 Module 1: Linux Basics & The Filesystem Hierarchy

Linux organizes everything into a single hierarchical tree structure starting from the root directory `/`. Unlike Windows, which uses drive letters (`C:`, `D:`), Linux represents physical disks, partitions, devices, and files as nodes under `/`.

```
                    / (Root Directory)
   ┌─────────┬─────────┼─────────┬─────────┬─────────┐
 /bin      /etc      /home     /var      /tmp    /scratch
(Binaries) (Config) (Users)   (Logs)   (Temp)   (High-Speed Storage)
```

### Key Directories to Remember:
* `/home/<username>`: Personal user workspace, configuration files (`.bashrc`), standard code projects.
* `/etc`: System-wide configuration files (e.g., `/etc/passwd`, `/etc/ssh/sshd_config`).
* `/tmp`: Temporary files cleared periodically or on reboot.
* `/var/log`: System and application logs.
* `/scratch` or `/work`: In HPC and cloud clusters, high-speed ephemeral parallel storage for massive datasets.

Let's set up our dedicated sandbox directory and explore basic navigation commands!

In [ ]:
%%bash
# 1.1 Create our dedicated training laboratory workspace
mkdir -p /tmp/gla_linux_lab/workspace
cd /tmp/gla_linux_lab/workspace

# Print current working directory
echo "=== Current Working Directory ==="
pwd

# Check Linux kernel and OS distribution details
echo -e "\n=== System & Kernel Information ==="
uname -srm
cat /etc/os-release | grep -E '^(NAME|VERSION)='

### 1.2 Creating, Moving, and Inspecting Files & Directories
Here are the essential file management utilities:
* `mkdir -p <dir>`: Create directories (including parent paths if they do not exist).
* `touch <filename>`: Create empty files or update file timestamps.
* `cp <source> <dest>`: Copy files (`-r` for recursive directory copies).
* `mv <source> <dest>`: Move or rename files/directories.
* `rm <file>`: Remove files (`rm -r` for directories, use with care!).
* `ls -lah`: List files with details (permissions, owner, size, modification date).

In [ ]:
%%bash
cd /tmp/gla_linux_lab/workspace

# Create a project folder structure
mkdir -p datasets logs scripts reports

# Create sample configuration and data files
touch datasets/train_sample.csv
touch datasets/test_sample.csv
touch scripts/run_pipeline.sh
touch config.yaml

# Write some sample content into files
echo "id,model_name,accuracy,loss" > datasets/results.csv
echo "1,ResNet50,0.924,0.076" >> datasets/results.csv
echo "2,BERT-Base,0.941,0.059" >> datasets/results.csv
echo "3,VisionTransformer,0.965,0.035" >> datasets/results.csv

# Inspect directory contents with human-readable sizes and permissions
echo "=== Directory Listing (ls -lah) ==="
ls -lah datasets/

### 1.3 Linux File Permissions (Read, Write, Execute)

Every file and directory in Linux has an access matrix with three permission types for three user categories:

| Target Scope | Notation | Meaning |
| :--- | :---: | :--- |
| **Owner (User)** | `u` | The creator/owner of the file |
| **Group** | `g` | Users in the file's primary group |
| **Others** | `o` | All other users on the system |

Permissions are represented in **Symbolic** (`rwx`) or **Octal** (Numeric) format:
* **Read (`r`)** = 4
* **Write (`w`)** = 2
* **Execute (`x`)** = 1
* **Sum total:** 7 = `rwx`, 6 = `rw-`, 5 = `r-x`, 4 = `r--`, 0 = `---`

Common permission sets:
* `chmod 755 script.sh` $\rightarrow$ Owner: `rwx` (7), Group: `r-x` (5), Others: `r-x` (5) (Standard for executable scripts)
* `chmod 644 data.txt` $\rightarrow$ Owner: `rw-` (6), Group: `r--` (4), Others: `r--` (4) (Standard for data files)
* `chmod 600 id_ed25519` $\rightarrow$ Owner: `rw-` (6), Group: `---` (0), Others: `---` (0) (Mandatory for SSH private keys)

In [ ]:
%%bash
cd /tmp/gla_linux_lab/workspace

# Create an automation script
cat << 'EOF' > scripts/system_audit.sh
#!/bin/bash
echo "System Hostname : $(hostname)"
echo "Current Date     : $(date)"
echo "Available Memory : $(free -h | awk '/Mem:/ {print $7}')"
EOF

# Check permissions BEFORE chmod
echo "=== Permissions BEFORE chmod ==="
ls -l scripts/system_audit.sh

# Grant execute permission (chmod 755)
chmod 755 scripts/system_audit.sh

echo -e "\n=== Permissions AFTER chmod 755 ==="
ls -l scripts/system_audit.sh

echo -e "\n=== Executing the Script ==="
./scripts/system_audit.sh

---
## ⚡ Module 2: Essential Commands, Text Processing & Pipelines

In Linux and Unix systems, the core philosophy is: **"Write programs that do one thing well, and connect them using streams."**

### 2.1 Standard Streams & Redirection
Every Linux program starts with three open standard streams:
1. `stdin` (Standard Input, file descriptor 0): Reads keyboard or piped input.
2. `stdout` (Standard Output, file descriptor 1): Displays normal program results.
3. `stderr` (Standard Error, file descriptor 2): Displays diagnostic and error messages.

Redirection Operators:
* `command > file`: Redirects `stdout` to `file` (overwrites).
* `command >> file`: Redirects `stdout` to `file` (appends).
* `command 2> errors.log`: Redirects `stderr` only to `errors.log`.
* `command > output.log 2>&1`: Redirects both `stdout` and `stderr` to `output.log`.
* `command 2> /dev/null`: Discards errors silently into the Linux "black hole" (`/dev/null`).

In [ ]:
%%bash
cd /tmp/gla_linux_lab/workspace

# 1. Demonstrating stdout vs stderr redirection
echo "Starting job simulation..." > logs/job_output.log

# Command producing normal output and error
ls datasets/ non_existent_folder_xyz > logs/success.log 2> logs/error.log || true

echo "=== Normal Output (logs/success.log) ==="
cat logs/success.log

echo -e "\n=== Error Stream Output (logs/error.log) ==="
cat logs/error.log

### 2.2 Text Processing Utilities: `cat`, `head`, `tail`, `grep`, `awk`
When dealing with logs, dataset headers, and cluster outputs, text manipulation is an everyday task:
* `cat <file>`: View entire file content.
* `head -n <N> <file>`: View first $N$ lines of a file.
* `tail -n <N> <file>`: View last $N$ lines of a file (use `tail -f` to follow live log growth).
* `grep "<pattern>" <file>`: Search for text matching pattern (`-i` case insensitive, `-v` invert match, `-c` count matches).
* `wc -l <file>`: Count lines in a file.
* `sort` and `uniq -c`: Sort lines and count unique occurrences.
* `awk '{print $1}'`: Extract specific columns/tokens from structured text.

Let's test these on a simulated cluster training log!

In [ ]:
%%bash
cd /tmp/gla_linux_lab/workspace

# Generate a synthetic HPC / Cloud training log
cat << 'EOF' > logs/cluster_training.log
2026-11-02 10:00:01 INFO [Node-01] Training epoch 1 started. LR=0.001
2026-11-02 10:00:15 WARNING [Node-03] GPU temperature reached 76C.
2026-11-02 10:01:02 INFO [Node-01] Epoch 1 completed. Loss: 0.452, Acc: 88.2%
2026-11-02 10:01:05 INFO [Node-02] Checkpoint saved: model_ep1.pt
2026-11-02 10:02:11 ERROR [Node-04] OutOfMemoryError: CUDA allocation failed on GPU:0
2026-11-02 10:02:12 WARNING [Node-04] Worker restarted after crash.
2026-11-02 10:02:30 INFO [Node-01] Training epoch 2 started. LR=0.00095
2026-11-02 10:03:45 ERROR [Node-02] Network socket timeout: dropped packet from Node-04
2026-11-02 10:04:10 INFO [Node-01] Epoch 2 completed. Loss: 0.312, Acc: 91.8%
2026-11-02 10:05:00 INFO [Node-01] Training completed successfully.
EOF

echo "=== Total Lines in Log ==="
wc -l logs/cluster_training.log

echo -e "\n=== Filtering All ERROR Events (grep 'ERROR') ==="
grep "ERROR" logs/cluster_training.log

echo -e "\n=== Powerful Pipeline: Count Log Level Occurrences (awk + sort + uniq) ==="
awk '{print $3}' logs/cluster_training.log | sort | uniq -c

---
## 🌐 Module 3: Environment Setup & Shell Variables

The shell uses **Environment Variables** to store system-wide and user-specific configurations, such as:
* `$USER`: Current logged-in user name.
* `$HOME`: Path to user's home directory.
* `$SHELL`: Path to the default shell binary (usually `/bin/bash`).
* `$PATH`: Colon-separated list of directories where the shell searches for executable commands.

### 3.1 Local vs Exported Variables
* A local variable (`VAR="value"`) is only accessible within the current shell session and is **not** inherited by child scripts or subshells.
* An exported variable (`export VAR="value"`) is propagated to all subshells, child processes, and scripts executed from that shell.

In [ ]:
%%bash
# 3.1 Exploring Built-in Environment Variables
echo "Current User : $USER"
echo "Home Directory: $HOME"
echo "Active Shell  : $SHELL"
echo "Current PWD   : $PWD"

# 3.2 Defining a local variable vs Exported variable
LOCAL_VAR="Local_GLA_Data"
export EXPORTED_VAR="Cluster_Master_Node_01"

# Verify inside a subshell using Python os.getenv
python3 -c "
import os
print('Subshell sees LOCAL_VAR    :', os.getenv('LOCAL_VAR'))
print('Subshell sees EXPORTED_VAR :', os.getenv('EXPORTED_VAR'))
"

### 3.2 Understanding `$PATH` and Shell Startup Files (`.bashrc`)

When you type `python` or `ls`, the shell doesn't search your entire hard drive. It looks sequentially through the list of directories listed in `$PATH`.

If you write custom automation tools, you can add their folder to `$PATH`!
* Temporary (current session): `export PATH="$PATH:/path/to/my/tools"`
* Permanent (every new terminal): Add the `export` line to `~/.bashrc` and run `source ~/.bashrc`.

In [ ]:
%%bash
cd /tmp/gla_linux_lab/workspace

# Create a custom binary in our scripts directory
cat << 'EOF' > scripts/mytool
#!/bin/bash
echo ">>> Congratulations! You invoked 'mytool' directly through \$PATH! <<<"
EOF
chmod +x scripts/mytool

# Add our custom directory to PATH
export PATH="$PATH:/tmp/gla_linux_lab/workspace/scripts"

# Now we can invoke 'mytool' directly without './scripts/' prefix!
which mytool
mytool

---
## 📜 Module 4: Shell Scripting for Automation (Basic to Medium)

A **Shell Script** is an executable text file containing a sequence of commands that the bash interpreter executes sequentially. It is the backbone of cluster automation, data pipeline orchestrations, and DevOps workflows.

### 4.1 Anatomy of a Production-Ready Bash Script
Every robust script has:
1. **Shebang (`#!/bin/bash`)**: Instructs the operating system which interpreter to use.
2. **Defensive Strict Mode (`set -euo pipefail`)**:
   - `set -e`: Exit immediately if any command exits with a non-zero (error) status.
   - `set -u`: Treat unset variables as an error and exit immediately.
   - `set -o pipefail`: Ensure a pipeline fails if any command in the chain fails (not just the last one).
3. **Special Positional Parameters:**
   - `$0`: Name of the script.
   - `$1`, `$2`, ... `$9`: First, second, etc., arguments passed to the script.
   - `$#`: Total count of arguments passed.
   - `$@`: All arguments as separate quoted strings.
   - `$?`: Exit code of the most recently executed command (`0` = Success, non-zero = Error).

In [ ]:
%%bash
cd /tmp/gla_linux_lab/workspace

cat << 'EOF' > scripts/demo_args.sh
#!/bin/bash
# Enable strict mode
set -euo pipefail

echo "=========================================="
echo "Script Name      (\$0) : $0"
echo "Total Arguments  (\$#) : $#"
echo "First Argument   (\$1) : ${1:-'None provided'}"
echo "Second Argument  (\$2) : ${2:-'None provided'}"
echo "All Arguments    (\$@) : $@"
echo "=========================================="

# Exit with code 0 (Success)
exit 0
EOF
chmod +x scripts/demo_args.sh

# Run script with sample parameters
./scripts/demo_args.sh "batch_job_42" "resnet50_model" "gpu_partition"
echo "Last Exit Code: $?"

### 4.2 Conditionals: `if`, `elif`, `else` and Test Operators
Bash allows checking conditions using the `[ ... ]` or `[[ ... ]]` syntax:

| Test Operator | Type | Meaning |
| :---: | :---: | :--- |
| `-f <file>` | File | True if file exists and is a regular file |
| `-d <dir>` | Directory | True if directory exists |
| `-z <str>` | String | True if string is empty (`length == 0`) |
| `-n <str>` | String | True if string is NOT empty |
| `"$A" == "$B"` | String | True if strings are identical |
| `-eq`, `-ne` | Numeric | Equal to, Not equal to |
| `-lt`, `-le` | Numeric | Less than, Less than or equal to |
| `-gt`, `-ge` | Numeric | Greater than, Greater than or equal to |

In [ ]:
%%bash
cd /tmp/gla_linux_lab/workspace

cat << 'EOF' > scripts/check_dataset.sh
#!/bin/bash
set -euo pipefail

TARGET_FILE="${1:-}"

# Check if user provided an argument
if [ -z "$TARGET_FILE" ]; then
    echo "⚠️ Error: Please specify a file path to inspect."
    echo "Usage: $0 <path_to_file>"
    exit 1
fi

# Check if file exists
if [ -f "$TARGET_FILE" ]; then
    LINE_COUNT=$(wc -l < "$TARGET_FILE")
    FILE_SIZE=$(du -h "$TARGET_FILE" | cut -f1)
    echo "✅ Success: File exists!"
    echo "   Path      : $TARGET_FILE"
    echo "   Size      : $FILE_SIZE"
    echo "   Line Count: $LINE_COUNT"
else
    echo "❌ Error: File '$TARGET_FILE' does not exist."
    exit 2
fi
EOF
chmod +x scripts/check_dataset.sh

# Test Case 1: Valid existing file
./scripts/check_dataset.sh "datasets/results.csv"

echo -e "\n--- Test Case 2: Non-existent file ---"
./scripts/check_dataset.sh "datasets/missing_data.csv" || true

### 4.3 Loops for Batch Processing & Automation
Loops allow you to iterate over lists of files, numbers, or command outputs:
* `for ... in ...; do ... done`: Best for known collections or glob patterns (`*.csv`).
* `while [ condition ]; do ... done`: Continues while the condition evaluates to true.

In [ ]:
%%bash
cd /tmp/gla_linux_lab/workspace

# Create some mock dataset splits
mkdir -p raw_data
for i in {1..5}; do
    echo "Sample data chunk $i" > "raw_data/partition_0$i.dat"
done

echo "=== Generated Files in raw_data/ ==="
ls -1 raw_data/

# Automation: Batch process and archive each file using a for loop
echo -e "\n=== Processing and Renaming Files ==="
for file in raw_data/*.dat; do
    base=$(basename "$file")
    line_count=$(wc -l < "$file")
    echo "Processing [$base] ... found $line_count lines."
done

### 4.4 Practical Automation Script: Automated Backup & Archive Utility
Here is a real-world, industry-standard script that creates a timestamped, gzip-compressed tar archive (`.tar.gz`) of a folder and generates a checksum verification manifest!

In [ ]:
%%bash
cd /tmp/gla_linux_lab/workspace

cat << 'EOF' > scripts/auto_backup.sh
#!/bin/bash
set -euo pipefail

SOURCE_DIR="${1:-datasets}"
BACKUP_DIR="${2:-backups}"
TIMESTAMP=$(date +"%Y%m%d_%H%M%S")
ARCHIVE_NAME="backup_${SOURCE_DIR//\//_}_${TIMESTAMP}.tar.gz"

echo "=========================================="
echo "🚀 AUTOMATED BACKUP UTILITY"
echo "   Source     : $SOURCE_DIR"
echo "   Destination: $BACKUP_DIR"
echo "   Timestamp  : $TIMESTAMP"
echo "=========================================="

# Ensure backup destination directory exists
mkdir -p "$BACKUP_DIR"

# Create compressed tarball
tar -czf "${BACKUP_DIR}/${ARCHIVE_NAME}" -C "$(dirname "$SOURCE_DIR")" "$(basename "$SOURCE_DIR")"

# Compute SHA-256 Checksum for data integrity
CHECKSUM=$(sha256sum "${BACKUP_DIR}/${ARCHIVE_NAME}" | awk '{print $1}')
echo "$CHECKSUM  $ARCHIVE_NAME" > "${BACKUP_DIR}/${ARCHIVE_NAME}.sha256"

echo "✅ Backup Completed Successfully!"
echo "   Archive File: ${BACKUP_DIR}/${ARCHIVE_NAME}"
echo "   Archive Size: $(du -h "${BACKUP_DIR}/${ARCHIVE_NAME}" | cut -f1)"
echo "   SHA-256     : $CHECKSUM"
echo "=========================================="
EOF
chmod +x scripts/auto_backup.sh

# Execute the backup script
./scripts/auto_backup.sh "datasets" "backups"

echo -e "\n=== Verifying Backup Files ==="
ls -lh backups/

---
## 🔐 Module 5: Secured Shell (SSH) Fundamentals

**Secured Shell (SSH)** is a cryptographic network protocol running on **TCP port 22** that allows secure, encrypted remote login, command execution, and file transfers over insecure networks.

```
       [ Client Laptop / Workstation ]
                      │
                      │  Encrypted Transport (TCP Port 22)
                      │  SSH-2.0 Protocol (Ed25519 / RSA)
                      ▼
       [ Remote HPC Cluster / Cloud Server ]
```

### 5.1 Symmetric vs Asymmetric Cryptography in SSH
* **Asymmetric Keypair Authentication:**
  - **Private Key (`id_ed25519`)**: Kept strictly confidential on your client computer. Permissions MUST be `chmod 600`. Never share it!
  - **Public Key (`id_ed25519.pub`)**: Placed on the remote server inside `~/.ssh/authorized_keys`. Safe to share with anyone.
* **Symmetric Session Encryption:** Once identity is proven, a high-speed symmetric session key (e.g., ChaCha20-Poly1305 or AES-GCM) encrypts the actual data traffic.

Let's generate a modern, high-security **Ed25519** SSH keypair!

In [ ]:
%%bash
# Create local ssh directory if it doesn't exist
mkdir -p ~/.ssh
chmod 700 ~/.ssh

# Generate a new Ed25519 SSH Keypair (no passphrase for automated lab demo)
ssh-keygen -t ed25519 -C "student_gla_hpc@gla.ac.in" -f ~/.ssh/gla_hpc_key -N "" -q

echo "=== SSH Directory Contents ==="
ls -la ~/.ssh/

echo -e "\n=== Public Key (Safe to share / Paste on Server) ==="
cat ~/.ssh/gla_hpc_key.pub

echo -e "\n=== Private Key (STRICTLY CONFIDENTIAL - chmod 600) ==="
ls -l ~/.ssh/gla_hpc_key

### 5.2 Passwordless Authentication: How `authorized_keys` Works
To log in without typing a password every time:
1. You copy your **Public Key** to the remote server.
2. The remote server appends it to `~/.ssh/authorized_keys`.
3. When connecting, the server creates a cryptographic challenge that can only be decrypted by your corresponding **Private Key**.

Let's simulate this mechanism right here on our Linux environment!

In [ ]:
%%bash
# Simulate authorizing our new key
cat ~/.ssh/gla_hpc_key.pub >> ~/.ssh/authorized_keys

# In Linux, SSH security strictly requires proper file permissions:
chmod 700 ~/.ssh
chmod 600 ~/.ssh/authorized_keys
chmod 600 ~/.ssh/gla_hpc_key

echo "=== Verifying authorized_keys file ==="
ls -la ~/.ssh/authorized_keys
echo "Keys installed: $(wc -l < ~/.ssh/authorized_keys) key(s)"
tail -n 1 ~/.ssh/authorized_keys

### 5.3 SSH Client Configuration (`~/.ssh/config`)
Instead of typing long commands every time:
```bash
ssh -i ~/.ssh/gla_hpc_key -p 2222 user123@cluster.hpc.gla.ac.in
```
You can create an SSH alias in `~/.ssh/config` so you simply type:
```bash
ssh gla-cluster
```

Let's construct a professional `~/.ssh/config` file!

In [ ]:
%%bash
# Create or append to ~/.ssh/config
cat << 'EOF' > ~/.ssh/config
# ==========================================
# GLA HPC & Cloud Cluster SSH Configuration
# ==========================================
Host gla-cluster
    HostName cluster.hpc.gla.ac.in
    User student01
    Port 22
    IdentityFile ~/.ssh/gla_hpc_key
    ServerAliveInterval 60
    ServerAliveCountMax 3

Host oneapi-cloud
    HostName devcloud.intel.com
    User u123456
    Port 22
    IdentityFile ~/.ssh/gla_hpc_key
EOF

chmod 600 ~/.ssh/config

echo "=== ~/.ssh/config Successfully Configured ==="
cat ~/.ssh/config

### 5.4 Secure Data Transfer: `scp` and `rsync`
Once SSH is configured, you can transfer files securely between machines:

1. **`scp` (Secure Copy):** Simple single or multi-file transfer.
   - Upload file to remote:
     ```bash
     scp -i ~/.ssh/gla_hpc_key my_model.pt student01@cluster.hpc.gla.ac.in:~/projects/
     ```
   - Download directory from remote:
     ```bash
     scp -r student01@cluster.hpc.gla.ac.in:~/results/ ./local_results/
     ```

2. **`rsync` (Remote Sync - Recommended for Big Data & Resuming):**
   - Synchronizes only changed bytes, supports compression (`-z`), preserves permissions (`-a`), and shows progress (`-P`):
     ```bash
     rsync -avzP datasets/ student01@cluster.hpc.gla.ac.in:~/datasets/
     ```

In [ ]:
%%bash
# Demonstrate rsync locally between two lab folders
mkdir -p /tmp/gla_linux_lab/source_data
mkdir -p /tmp/gla_linux_lab/dest_data

# Create test data
echo "Payload data file 1" > /tmp/gla_linux_lab/source_data/file1.txt
echo "Payload data file 2" > /tmp/gla_linux_lab/source_data/file2.txt

# Sync using rsync archive mode (-a) and verbose output (-v)
echo "=== Running rsync sync ==="
rsync -av /tmp/gla_linux_lab/source_data/ /tmp/gla_linux_lab/dest_data/

echo -e "\n=== Destination Directory After Sync ==="
ls -la /tmp/gla_linux_lab/dest_data/

---
## 🏆 Module 6: Interactive Student Laboratory Challenges

Test your understanding with these three hands-on challenges. Write the solution in the code cell below each task and execute it to verify your result!

---
### 🧩 Challenge 1: Log Diagnostic Analyzer
**Scenario:** A cluster training run dumped server logs into `/tmp/gla_linux_lab/challenge_logs/server.log`.  
**Task:**
1. Create the log with the provided snippet.
2. Write a single pipeline using `grep`, `awk`, `sort`, or `wc` to count how many **`CRITICAL`** and **`WARNING`** messages occurred.
3. Save the summary into `report_summary.txt`.

In [ ]:
%%bash
# Setup challenge data
mkdir -p /tmp/gla_linux_lab/challenge_logs
cat << 'EOF' > /tmp/gla_linux_lab/challenge_logs/server.log
2026-11-02 12:00:01 INFO [Node-01] Health check OK.
2026-11-02 12:00:10 WARNING [Node-02] High RAM usage (89%).
2026-11-02 12:00:15 CRITICAL [Node-03] Disk full on /scratch (100%).
2026-11-02 12:00:22 WARNING [Node-01] Network latency spike (120ms).
2026-11-02 12:00:30 CRITICAL [Node-03] Process terminated: out of disk space.
2026-11-02 12:00:45 INFO [Node-04] Task finished normally.
EOF

# ==========================================
# STUDENT SOLUTION CODE:
# Filter WARNING and CRITICAL lines and save count
# ==========================================
echo "=== Summary of High Priority Incidents ==="
grep -E "(WARNING|CRITICAL)" /tmp/gla_linux_lab/challenge_logs/server.log | awk '{print $3}' | sort | uniq -c | tee /tmp/gla_linux_lab/challenge_logs/report_summary.txt

echo -e "\n=== Verification ==="
test -f /tmp/gla_linux_lab/challenge_logs/report_summary.txt && echo "🎉 Challenge 1 PASSED! Report generated successfully." || echo "❌ Failed to create report."

---
### 🧩 Challenge 2: Automated File Organizer Script
**Scenario:** You have a directory with mixed file types (`.csv`, `.log`, `.sh`, `.txt`).  
**Task:**
Write a bash script that iterates over all files in the directory and automatically organizes them into subfolders based on extension:
- `.csv` files $\rightarrow$ `data/`
- `.log` files $\rightarrow$ `logs/`
- `.sh` files $\rightarrow$ `scripts/`

In [ ]:
%%bash
# Setup mixed files
mkdir -p /tmp/gla_linux_lab/mixed_files
cd /tmp/gla_linux_lab/mixed_files
touch run_test.sh train_metrics.csv debug.log evaluate.sh validation.csv error.log notes.txt

echo "=== Files BEFORE Organization ==="
ls -1

# ==========================================
# STUDENT SOLUTION CODE:
# Organize files by extension
# ==========================================
mkdir -p data logs scripts other

for f in *; do
    if [ -f "$f" ]; then
        case "$f" in
            *.csv) mv "$f" data/ ;;
            *.log) mv "$f" logs/ ;;
            *.sh)  mv "$f" scripts/ ;;
            *)     mv "$f" other/ ;;
        esac
    fi
done

echo -e "\n=== Organized Directory Structure ==="
ls -R /tmp/gla_linux_lab/mixed_files/

echo -e "\n=== Verification ==="
if [ -d data ] && [ -d logs ] && [ -d scripts ]; then
    echo "🎉 Challenge 2 PASSED! Files successfully categorized."
fi

---
### 🧩 Challenge 3: SSH Security Key Verification
**Scenario:** As a cluster administrator, you must verify that user SSH keys adhere to the cluster security policy:
1. `~/.ssh` directory must have permissions `700` (`drwx------`).
2. Private keys must have permissions `600` (`-rw-------`).
3. An authorized_keys entry must be present.  

Run the automated compliance checker below to confirm your setup!

In [ ]:
%%bash
# ==========================================
# SSH SECURITY AUDIT SCRIPT
# ==========================================
echo "🔍 Running SSH Security Compliance Audit..."

# Check ~/.ssh permissions
SSH_DIR_PERM=$(stat -c "%a" ~/.ssh)
if [ "$SSH_DIR_PERM" -eq 700 ]; then
    echo "  ✅ ~/.ssh directory permissions: $SSH_DIR_PERM (Compliant)"
else
    echo "  ⚠️ Warning: ~/.ssh is $SSH_DIR_PERM, fixing to 700..."
    chmod 700 ~/.ssh
fi

# Check private key permissions
KEY_FILE="$HOME/.ssh/gla_hpc_key"
if [ -f "$KEY_FILE" ]; then
    KEY_PERM=$(stat -c "%a" "$KEY_FILE")
    if [ "$KEY_PERM" -eq 600 ]; then
        echo "  ✅ Private key ($KEY_FILE) permissions: $KEY_PERM (Compliant)"
    else
        echo "  ⚠️ Warning: Private key is $KEY_PERM, fixing to 600..."
        chmod 600 "$KEY_FILE"
    fi
fi

# Check authorized_keys
if [ -f "$HOME/.ssh/authorized_keys" ]; then
    AUTH_KEYS_COUNT=$(wc -l < "$HOME/.ssh/authorized_keys")
    echo "  ✅ authorized_keys file active ($AUTH_KEYS_COUNT key entries)"
fi

echo -e "\n🎉 Challenge 3 PASSED! SSH configuration is 100% compliant with HPC security standards."

---
## 🏁 Summary & Key Takeaways

| Topic | Core Commands / Concepts | Key Best Practice |
| :--- | :--- | :--- |
| **Linux Basics** | `pwd`, `cd`, `ls -lah`, `mkdir -p`, `cp`, `mv`, `rm` | Know your filesystem (`/home` vs ephemeral `/scratch`). |
| **Permissions** | `chmod 755`, `chmod 600`, `chmod 644`, `chown` | Always secure private keys (`600`) and scripts (`755`). |
| **Pipelines & Redirection** | `>`, `>>`, `2>&1`, `|`, `grep`, `awk`, `sort`, `uniq` | Combine small, specialized Unix tools to process logs and streams. |
| **Environment Setup** | `export VAR="val"`, `$PATH`, `~/.bashrc`, `source` | Add custom tool directories to `$PATH` for seamless CLI access. |
| **Shell Scripting** | `#!/bin/bash`, `set -euo pipefail`, `$1`, `$#`, `if`, `for` | Always use strict mode (`-euo pipefail`) to avoid silent failures. |
| **Secured Shell (SSH)** | `ssh-keygen -t ed25519`, `~/.ssh/config`, `scp`, `rsync` | Use Ed25519 keys, passwordless auth, and SSH config aliases. |

---
**Course:** High Performance Computing & Distributed AI Masterclass  
**Module:** Day 01 — Linux Basics, Shell Scripting & Secured Shell  
**Department:** Computer Engineering & Applications, GLA University  
*Happy Computing & Shell Scripting!* 🚀